## Оптимизация выполнения кода, векторизация, Numba

Материалы:
* Лекция: Оптимизация выполнения кода, векторизация, Numba
* IPython Cookbook, Second Edition (2018), глава 4
* https://numba.pydata.org/numba-doc/latest/user/5minguide.html

## Задачи для совместного разбора

1. Сгенерируйте массив `A` из `N=1млн` случайных целых чисел на отрезке от 0 до 1000. Пусть `B[i] = A[i] + 100`. Посчитайте среднее значение массива `B`.

In [6]:
import numpy as np

In [7]:
A = np.random.randint(0, 1000, size=(1000000,))

In [8]:
from numba import njit

def f1(A):
    acc, cnt = 0, 0
    for x in A:
        acc += (x + 100)
        cnt += 1
    return acc / cnt

def f2(A):
    acc = 0
    for x in A:
        acc += (x + 100)
    return acc / len(A)

def f3(A):
    acc = 0
    for x in A:
        acc += x
    return acc / len(A) + 100

@njit
def f4(A):
    acc, cnt = 0, 0
    for x in A:
        acc += (x + 100)
        cnt += 1
    return acc / cnt

@njit
def f5(A):
    acc = 0
    for x in A:
        acc += x
    return acc / len(A) + 100

In [9]:
%timeit f1(A)

199 ms ± 11.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [10]:
%timeit f2(A)

205 ms ± 55.9 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [11]:
%timeit f3(A)

110 ms ± 19.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [12]:
%timeit f4(A)

668 µs ± 154 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [13]:
%timeit f5(A)

466 µs ± 7.09 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


2. Создайте таблицу 2млн строк и с 4 столбцами, заполненными случайными числами. Добавьте столбец `key`, которые содержит элементы из множества английских букв. Выберите из таблицы подмножество строк, для которых в столбце `key` указаны первые 5 английских букв.

In [14]:
import pandas as pd
import string
N = 2_000_000
df = pd.DataFrame(np.random.randn(N, 4), columns=[f'col{i}' for i in range(4)])
df['key'] = np.random.choice(list(string.ascii_letters.lower()), N, replace=True)

In [15]:
def g1(df):
    res = pd.DataFrame()
    for letter in ['a', 'b', 'c', 'd', 'e']:
        res = pd.concat([res, df[df['key']==letter]], axis=0)
    return res

def g2(df):
    res = pd.concat([df[df['key']==letter] for letter in ['a', 'b', 'c', 'd', 'e']], axis=0)
    return res

@njit
def g3(df): # ERROR
    res = pd.DataFrame()
    for letter in ['a', 'b', 'c', 'd', 'e']:
        res = pd.concat([res, df[df['key']==letter]], axis=0)
    return res

def g4(df):
    res = df[df['key'].isin(['a', 'b', 'c', 'd', 'e'])]
    return res

In [16]:
%timeit g1(df)

924 ms ± 121 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [17]:
%timeit g2(df)

1.06 s ± 335 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [18]:
%timeit g4(df)

119 ms ± 23.7 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


## Лабораторная работа

In [19]:
!pip install line_profiler

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 19.5 MB/s eta 0:00:00


1. В файлах `recipes_sample.csv` и `reviews_sample.csv` (__ЛР 2__) находится информация об рецептах блюд и отзывах на эти рецепты соответственно. Загрузите данные из файлов в виде `pd.DataFrame` с названиями `recipes` и `reviews`. Обратите внимание на корректное считывание столбца(ов) с индексами. Приведите столбцы к нужным типам.

Реализуйте несколько вариантов функции подсчета среднего значения столбца `rating` из таблицы `reviews` для отзывов, оставленных в 2010 году.

A. С использованием метода `DataFrame.iterrows` исходной таблицы;

Б. С использованием метода `DataFrame.iterrows` таблицы, в которой сохранены только отзывы за 2010 год;

В. С использованием метода `Series.mean`.

Проверьте, что результаты работы всех написанных функций корректны и совпадают. Измерьте выполнения всех написанных функций.


In [1]:
import numpy as np
import pandas as pd
from numba import njit

# Загрузка данных
recipes = pd.read_csv('recipes_sample.csv', index_col=0)
reviews = pd.read_csv('reviews_sample.csv', index_col=0)

# Приведение типов
reviews['date'] = pd.to_datetime(reviews['date'])
reviews['rating'] = reviews['rating'].astype(float)

# A. iterrows по всей таблице
def avg_rating_iterrows_full(df):
    total, count = 0, 0
    for idx, row in df.iterrows():
        if row['date'].year == 2010:
            total += row['rating']
            count += 1
    return total / count if count > 0 else 0

# Б. iterrows по отфильтрованной таблице
def avg_rating_iterrows_filtered(df):
    df_2010 = df[df['date'].dt.year == 2010]
    total, count = 0, 0
    for idx, row in df_2010.iterrows():
        total += row['rating']
        count += 1
    return total / count if count > 0 else 0

# В. Series.mean
def avg_rating_mean(df):
    return df[df['date'].dt.year == 2010]['rating'].mean()

print(avg_rating_iterrows_full(reviews))
print(avg_rating_iterrows_filtered(reviews))
print(avg_rating_mean(reviews))

%timeit avg_rating_iterrows_full(reviews)
%timeit avg_rating_iterrows_filtered(reviews)
%timeit avg_rating_mean(reviews)

4.4544402182900615
4.4544402182900615
4.4544402182900615
6.1 s ± 500 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
525 ms ± 17.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
11.7 ms ± 3.95 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


2. Какая из созданных функций выполняется медленнее? Что наиболее сильно влияет на скорость выполнения? Для ответа использовать профайлер `line_profiler`. Сохраните результаты работы профайлера в отдельную текстовую ячейку и прокомментируйте результаты его работы.

(*). Сможете ли вы ускорить работу функции 1Б, отказавшись от использования метода `iterrows`, но не используя метод `mean`?

In [ ]:
# Медленнее всего выполняется функция A (iterrows по всей таблице)
# На скорость влияет: фильтрация до итерации (Б быстрее А),
# использование векторизованного mean (В быстрее всего)

In [20]:
import line_profiler
import pandas as pd
import numpy as np
reviews = pd.read_csv('reviews_sample.csv', index_col=0)
reviews['date'] = pd.to_datetime(reviews['date'])

def avg_rating_iterrows_full(df):
    total, count = 0, 0
    for idx, row in df.iterrows():
        if row['date'].year == 2010:
            total += row['rating']
            count += 1
    return total / count if count > 0 else 0

def avg_rating_iterrows_filtered(df):
    df_2010 = df[df['date'].dt.year == 2010]
    total, count = 0, 0
    for idx, row in df_2010.iterrows():
        total += row['rating']
        count += 1
    return total / count if count > 0 else 0

lp = line_profiler.LineProfiler()
lp_wrapper = lp(avg_rating_iterrows_full)
lp_wrapper(reviews)
lp.print_stats()

lp = line_profiler.LineProfiler()
lp_wrapper = lp(avg_rating_iterrows_filtered)
lp_wrapper(reviews)
lp.print_stats()

Timer unit: 1e-09 s

Total time: 32.1678 s
File: /tmp/ipykernel_2883/1497364653.py
Function: avg_rating_iterrows_full at line 7

Line #      Hits         Time  Per Hit   % Time  Line Contents
     7                                           def avg_rating_iterrows_full(df):
     8         1       4404.0   4404.0      0.0      total, count = 0, 0
     9    126697     2.84e+10 224253.2     88.3      for idx, row in df.iterrows():
    10    126696 3488860449.0  27537.3     10.8          if row['date'].year == 2010:
    11     12094  259945903.0  21493.8      0.8              total += row['rating']
    12     12094    6782173.0    560.8      0.0              count += 1
    13         1       2724.0   2724.0      0.0      return total / count if count > 0 else 0

Timer unit: 1e-09 s

Total time: 2.62265 s
File: /tmp/ipykernel_2883/1497364653.py
Function: avg_rating_iterrows_filtered at line 15

Line #      Hits         Time  Per Hit   % Time  Line Contents
    15                            

In [2]:
def avg_rating_fast(df):
    mask = df['date'].dt.year == 2010
    return df.loc[mask, 'rating'].sum() / mask.sum()

%timeit avg_rating_fast(reviews)

8.18 ms ± 362 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


3. Вам предлагается воспользоваться функцией, которая собирает статистику о том, сколько отзывов содержат то или иное слово. Измерьте время выполнения этой функции. Сможете ли вы найти узкие места в коде, используя профайлер? Выпишите (словами), что в имеющемся коде реализовано неоптимально. Оптимизируйте функцию и добейтесь значительного (как минимум, на один порядок) прироста в скорости выполнения.

In [21]:
from collections import Counter

def get_word_reviews_count_old(df):
    word_reviews = {}
    for _, row in df.dropna(subset=['review']).iterrows():
        recipe_id, review = row['recipe_id'], row['review']
        words = review.split(' ')
        for word in words:
            if word not in word_reviews:
                word_reviews[word] = []
            word_reviews[word].append(recipe_id)
    word_reviews_count = {}
    for _, row in df.dropna(subset=['review']).iterrows():
        review = row['review']
        words = review.split(' ')
        for word in words:
            word_reviews_count[word] = len(word_reviews[word])
    return word_reviews_count

def get_word_reviews_count_new(df):
    word_count = {}
    for review in df.dropna(subset=['review'])['review']:
        for word in review.split():
            word_count[word] = word_count.get(word, 0) + 1
    return word_count

def get_word_reviews_count_fastest(df):
    text = ' '.join(df.dropna(subset=['review'])['review'])
    return dict(Counter(text.split()))

%timeit get_word_reviews_count_old(reviews)
%timeit get_word_reviews_count_new(reviews)
%timeit get_word_reviews_count_fastest(reviews)

22.3 s ± 1.36 s per loop (mean ± std. dev. of 7 runs, 1 loop each)
2.1 s ± 291 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
2.35 s ± 387 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


4. Напишите несколько версий функции `MAPE` (см. [MAPE](https://en.wikipedia.org/wiki/Mean_absolute_percentage_error)) для расчета среднего абсолютного процентного отклонения значения рейтинга отзыва на рецепт от среднего значения рейтинга по всем отзывам для этого рецепта.
    1. Без использования векторизованных операций и методов массивов `numpy` и без использования `numba`
    2. Без использования векторизованных операций и методов массивов `numpy`, но с использованием `numba`
    3. С использованием векторизованных операций и методов массивов `numpy`, но без использования `numba`
    4. C использованием векторизованных операций и методов массивов `numpy` и `numba`
    
Измерьте время выполнения каждой из реализаций.

Замечание: удалите из выборки отзывы с нулевым рейтингом.


In [22]:
reviews_no_zero = reviews[reviews['rating'] > 0].copy()
mean_by_recipe = reviews_no_zero.groupby('recipe_id')['rating'].transform('mean')

def mape_pure_python(df, means):
    total, count = 0, 0
    for i in range(len(df)):
        actual = df['rating'].iloc[i]
        pred = means.iloc[i]
        if actual > 0:
            total += abs((actual - pred) / actual)
            count += 1
    return (total / count) * 100

@njit
def mape_numba(actual, pred):
    total, count = 0.0, 0
    for i in range(len(actual)):
        if actual[i] > 0:
            total += abs((actual[i] - pred[i]) / actual[i])
            count += 1
    return (total / count) * 100

def mape_vectorized(df, means):
    actual = df['rating'].values
    pred = means.values
    mask = actual > 0
    return np.mean(np.abs((actual[mask] - pred[mask]) / actual[mask])) * 100

@njit
def mape_vectorized_numba(actual, pred):
    mask = actual > 0
    return np.mean(np.abs((actual[mask] - pred[mask]) / actual[mask])) * 100

print(mape_pure_python(reviews_no_zero, mean_by_recipe))
print(mape_numba(reviews_no_zero['rating'].values, mean_by_recipe.values))
print(mape_vectorized(reviews_no_zero, mean_by_recipe))
print(mape_vectorized_numba(reviews_no_zero['rating'].values, mean_by_recipe.values))

%timeit mape_pure_python(reviews_no_zero, mean_by_recipe)
%timeit mape_numba(reviews_no_zero['rating'].values, mean_by_recipe.values)
%timeit mape_vectorized(reviews_no_zero, mean_by_recipe)
%timeit mape_vectorized_numba(reviews_no_zero['rating'].values, mean_by_recipe.values)

11.17155025905884
11.17155025905884
11.171550259058085
11.17155025905884
1.55 s ± 333 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
215 µs ± 7.55 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)
1.45 ms ± 238 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)
1.01 ms ± 177 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)
